# Computer Graphics

**Computer graphics** is the field dedicated to generating, manipulating, and synthesising visual content with computers.

The primary goal is to transform an abstract description of a scene (shapes, materials, lights, camera) into a compelling 2D image.

## The rendering pipeline

The rendering pipeline describes how a 3D scene is converted into a 2D image on the screen. These stages reflect the hardware-level process used by the GPU.

The stages are:

1. **Object transformation** - converts object coordinates from local space to world space
3. **Clipping** - removes geometry outside the view
4. **Backface culling** - discards polygons facing away from the camera
5. **Rasterisation** - determines which screen pixels are covered by triangles, producing fragments (potential pixels)
6. **Hidden surface removal (HSR) & Shading** - depth testing to keep only the closest fragment per pixel and computation of final colour

### Object transformation

Object transformation is the process of taking a model defined in its own coordinate system (local space) and placing it correctly into the scene (world space).

In the local space, each object is defined relative to its own centre (origin). 

- This essentially provides a template or blueprint for the object, which can then be reused easily


In the global space, all objects are placed into a shared global coordinate system, so positions are relative to the world origin.

- Multiple copies of the same object can be placed in different locations, and objects can interact with other objects in the scene

The relation between the local and global space is given by:
$$
P_{world} = M \times P_{local}
$$
where:

- $P_{local}$ is the vertex in object (local) space
- $M$ is the transformation matrix (model matrix)
- $P_{world}$ is the vertex in world space

$M$ is a $4 \times 4$ matrix which combines translation, rotation and scaling. The position of each vertex is calculated independently, using SIMD (Single Instruction, Multiple Data) on the GPU  to transform thousands of vertices in parallel (at the same time).

Vertices are represented in homogeneous coordinates as $(x,y,x,w)$ where $w=1$ for positions, allowing translation to be included in matrix multiplication, while direction vectors use $w=0$ so they are unaffected by translation. After applying the projection matrix, 
$w$ encodes depth-related information of the vertex.

### Perspective projection

Perspective projection is the process of simulating a camera lens by transforming 3D coordinates into a 2D representation, where distant objects appear smaller.

To render a 3D scene:

- a **virtual camera** is defined, which establishes the point from which the scene is being viewed
- this camera defines a specific volume of space called the **view frustum**
  - the view frustum is a truncated pyramid that determines exactly what is visible
  - only objects that fall inside the view frustum are rendered
  - the view frustum is limited by:
    - Field of View (FOV)
    - the Near Plane (closest visible distance)
    - the Far  Plane (draw distance)
- a **projection matrix** transforms coordinates into **clip space**, mapping the view frustum into a unit cube
- a **perspective divide** is applied after clipping, where the $x$, $y$, and $z$ coordinates are divided by the homogeneous coordinate $w$, which encodes depth information generated by the projection matrix
  - this causes objects further away to appear smaller, creating the illusion of depth

![Perspective Projection](perspective_projection.png)

### Clipping

Clipping is the process of removing geometry that lies outside the camera's view (view frustum) to reduce the amount of processing needed. 

The GPU checks whether vertices/primitives lie within the viewing volume, keeping only those that are within the view: 

- if a triangle is partly inside and partly outside the view frustum, it cannot be discarded (as part is visible) but it cannot be rendered in full (as part is outside the view)
- the solution is to cut the triangle along the clipping boundary and create new vertices where edges intersect the frustum, which forms new triangles which fit entirely within the view volume
  - a common algorithm that does this is Sutherland-Hodgman

![Clipping](clipping.png)

Clipping is performed after the projection matrix is applied and before the perspective divide, as perspective projection transforms the view frustum into a unit cube in clip space. This makes it easier to check if points lie within the cube; simply check that:

- $-w \le x \le w$
- $-w \le y \le w$
- $-w \le z \le w$

### Backface culling

Backface culling is the process of discarding polygons that face away from the camera, since they are not visible in a closed 3D object, so rendering them is computationally wasteful. 

To determine if a surface is facing the camera, a dot product can be used:
$$ \vec{V} \cdot \vec{N}$$
where:
- $\vec{V}$ is the view direction
- $\vec{N}$ is the surface normal vector

If the dot product is positive, the face is pointing away, so the surface is discarded. If the dot product is negative, the face is pointing towards the camera, and is kept. 

A slightly more efficient way of determining whether triangles are back-facing is by checking **winding order** after projection:

- winding order determines whether a triangle is front- or back-facing by checking whether its vertices appear clockwise or counter-clockwise in screen space
- one winding direction is defined as front-facing and the opposite is treated as back-facing, allowing fast culling without computing normals

### Rasterisation

Rasterisation is the process of converting continuous 2D geometric shapes (triangles) into discrete screen fragments (potential pixels).

After clipping and culling, geometry is still made of mathematical lines and triangles. However, the screen is a grid of pixels. 

Rasterisation uses a **scanline algorithm** to determine which pixels are covered by each triangle:

- triangle vertices are sorted by $y$-coordinate to find the top, middle and bottom
- calculate the equation for the left and right edges
  - these are typically linear, in the form $Ax+By+C=0$
- for each horizontal row (scanline) in the pixel grid,
  - compute:
    - $x_{start}$: the x-coordinate where the scanline intersects the left edge
    - $x_{end}$: the x-coordinate where the scanline intersects the right edge
  - the pixels from $x_{start}$ to $x_{end}$ form a **span**
    - each pixel centre inside the span becomes a fragment
    - a fragment is a potential pixel, containing screen-space position $(x, y)$, depth ($z$-value) and attributes such as colour and texture
      - attributes are linearly interpolated between $x_{start}$ and $x_{end}$ to compute per-fragment values
  - generally, a pixel is only filled if its centre point lies inside the triangle

![Rasterisation](rasterisation.png)

Because pixels are discrete, triangle edges look jagged. This is called **aliasing**. It occurs because continuous shapes are approximated using a discrete pixel grid.

### Hidden surface removal (HSR) and shading

The final stage of the rendering pipeline determines:

- Which fragments are visible (Hidden Surface Removal)
- What colour they should be (Shading)

#### Shading

Shading is performed by the **fragment shader**. 

The fragment shader calculates the final colour of each fragment using:

- surface normals
- lighting (light sources, direction, intensity)
- textures (UV mapping)

For each fragment, it outputs a final pixel colour.

#### Hidden surface removal

Not all fragments should be drawn - some are hidden behind others.

##### Painter's algorithm

The historical approach was the Painter's algorithm:

- sort polygons by depth (approximate back-to-front ordering)
- draw far objects first, then paint nearer ones on top

However, this method had problems:

- it was slow, since sorting is $O(n \log n)$
- it fails completely for cyclic overlap, where three triangles overlap each other in a cycle (A in front of B, B in front of C, C in front of A)

##### $Z$-buffer

The modern solution is to use a **$Z$-buffer** (**depth buffer**):

- the GPU maintains the depth buffer, which is a 2D array of floats matching the screen resolution that stores the depth of the closest fragment seen so far
- for each fragment at position $(x,y)$:
  - if $Z_{new} < Z_{stored}$:
    - update depth buffer with $Z_{new}$
    - write fragment to frame buffer (stores final pixel colour output of the image)
  - else, discard the fragment (as it is hidden)

![Painter's algorithm and Z-buffer](hsr.png)

This method is:

- fast - constant ($O(1)$) check time per pixel
- order independent - triangles can be processed in any order
- precise - visibility is decided per pixel (can handle cyclic overlaps)
- highly parallelisable - depth testing is performed independently per fragment, making it ideal for GPU architecture

However, a possible problem can occur if the depth buffer has limited precision:

- **Z-fighting** is when the GPU cannot reliably distinguish which fragment is closer, causing flickering